In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Paths for the processed data and TSFRESH features
processed_train_path = 'data/processed/train_transformed_combined.csv'
processed_test_path = 'data/processed/test_transformed_combined.csv'
tsfresh_train_path = 'data/tsfresh/train_combined_all_features_filled.csv'
tsfresh_test_path = 'data/tsfresh/test_combined_all_features_filled.csv'

# Load data
train_combined = pd.read_csv(processed_train_path, index_col='Date', parse_dates=True)
test_combined = pd.read_csv(processed_test_path, index_col='Date', parse_dates=True)
tsfresh_features_train = pd.read_csv(tsfresh_train_path, index_col='Date', parse_dates=True)
tsfresh_features_test = pd.read_csv(tsfresh_test_path, index_col='Date', parse_dates=True)

# Selected targets and their respective features
target_features = {
    'FEDFUNDS': ['UNRATE__change_quantiles__f_agg_"mean"__isabs_True__qh_0.6__ql_0.2'],
    'CPIAUCSL': [
        'PCEPI__fft_coefficient__attr_"angle"__coeff_0', 
        'CUSR0000SAH1__large_standard_deviation__r_0.25',
        'CUMFNS__large_standard_deviation__r_0.30000000000000004',
        'CPILFESL__large_standard_deviation__r_0.30000000000000004',
        'DGS10__longest_strike_below_mean',
        'HOUST__number_peaks__n_50',
        'CUMFNS__has_duplicate_min',
        'BAA__sum_of_reoccurring_values',
        'CPIAUCSL__large_standard_deviation__r_0.4'
    ],
    'PRFI': ['PCEPI__longest_strike_below_mean']
}

# Model parameters
xgboost_params = {'max_depth': 9, 'learning_rate': 0.0168, 'n_estimators': 193, 'min_child_weight': 3, 'subsample': 0.5866, 'colsample_bytree': 0.9177, 'reg_alpha': 0.0125, 'reg_lambda': 8.577e-05, 'verbosity': 0}
lightgbm_params = {'num_leaves': 119, 'learning_rate': 0.0832, 'n_estimators': 71, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.7402, 'colsample_bytree': 0.9405, 'reg_alpha': 0.000756, 'reg_lambda': 0.000259, 'verbosity': -1}

# Function to compute MSE
def compute_mse(X_train, X_test, y_train, y_test, features):
    X_train = X_train[features].dropna()
    X_test = X_test[features].dropna()
    y_train = y_train.dropna()
    y_test = y_test.dropna()

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    mse_scores = {}

    # XGBoost
    xgb_model = XGBRegressor(**xgboost_params)
    xgb_model.fit(X_train_scaled, y_train)
    y_pred_xgb = xgb_model.predict(X_test_scaled)
    mse_scores['XGBoost'] = mean_squared_error(y_test, y_pred_xgb)

    # LightGBM
    lgb_model = LGBMRegressor(**lightgbm_params)
    lgb_model.fit(X_train_scaled, y_train)
    y_pred_lgb = lgb_model.predict(X_test_scaled)
    mse_scores['LightGBM'] = mean_squared_error(y_test, y_pred_lgb)

    return mse_scores

# Evaluate the selected targets
results = {}
for target, features in target_features.items():
    print(f"\nEvaluating target: {target}")
    X_train = tsfresh_features_train[features]
    X_test = tsfresh_features_test[features]
    y_train = train_combined[[target]].dropna()
    y_test = test_combined[[target]].dropna()

    mse_scores = compute_mse(X_train, X_test, y_train, y_test, features)
    results[target] = mse_scores
    print(f"XGBoost MSE: {mse_scores['XGBoost']}")
    print(f"LightGBM MSE: {mse_scores['LightGBM']}")

# Display results
print("\nFinal MSE Results:")
for target, mse in results.items():
    print(f"Target: {target}, XGBoost MSE: {mse['XGBoost']}, LightGBM MSE: {mse['LightGBM']}")
